# AdaBoost — boosting by reweighting hard examples

> Tutorial pair for [`adaboost.py`](adaboost.py).

## 1. Intuition
Train a sequence of *weak* learners (decision stumps). After each one, **increase
the weight** of the examples it got wrong, so the next learner focuses on the
hard cases. Give each learner a vote $\alpha_m$ proportional to how well it did.
The committee of reweighted stumps becomes a strong classifier. AdaBoost is, in
disguise, stagewise additive modeling under the **exponential loss**.

## 2. Concept (the slide)
- **Weak learner:** a decision stump (depth-1 tree) — barely better than chance.
- **Sample weights** $w_i$: start uniform; after round $m$, up-weight
  misclassified points by $e^{\alpha_m}$ and renormalize.
- **Learner weight** $\alpha_m$: large when the weighted error $\varepsilon_m$ is
  small; can go negative if worse than random (we stop then).
- **Final vote:** $H(x)=\arg\max_k \sum_m \alpha_m\,\mathbb 1[h_m(x)=k]$.
- **SAMME** generalizes the two-class rule to $K$ classes with a $\log(K-1)$ term.

## 3. Math derivation

**Exponential loss & forward stagewise additive modeling.** Consider binary
labels $y\in\{-1,+1\}$ and an additive score $F_M(x)=\sum_{m} \alpha_m h_m(x)$
with $h_m(x)\in\{-1,+1\}$. AdaBoost greedily minimizes the **exponential loss**

$$\mathcal L=\sum_{i=1}^n \exp\!\big(-y_i F_M(x_i)\big).$$

At round $m$, with $F_{m-1}$ fixed, define weights
$w_i^{(m)}=\exp(-y_i F_{m-1}(x_i))$. We choose $(\alpha,h)$ to minimize

$$\sum_i w_i^{(m)} e^{-\alpha y_i h(x_i)}
 = e^{-\alpha}\!\!\sum_{y_i=h(x_i)}\!\! w_i^{(m)}
 + e^{\alpha}\!\!\sum_{y_i\ne h(x_i)}\!\! w_i^{(m)} .$$

Let the **weighted error** be
$\varepsilon_m=\dfrac{\sum_i w_i^{(m)}\,\mathbb 1[h(x_i)\ne y_i]}{\sum_i w_i^{(m)}}$.
The bracket is $(e^{\alpha}-e^{-\alpha})\varepsilon_m + e^{-\alpha}$. For fixed
$\alpha>0$ this is minimized by the $h$ with the **smallest** $\varepsilon_m$
(so we just fit a weighted-error stump). Setting $\partial/\partial\alpha=0$:

$$-e^{-\alpha}(1-\varepsilon_m)+e^{\alpha}\varepsilon_m=0
 \;\Rightarrow\;
 \boxed{\;\alpha_m=\tfrac12\log\frac{1-\varepsilon_m}{\varepsilon_m}\;}$$

(the factor $\tfrac12$ or $1$ is a convention absorbed into the vote scale).
Substituting back, the weight update is
$w_i^{(m+1)}=w_i^{(m)}e^{-\alpha_m y_i h_m(x_i)}$, i.e. **multiply misclassified
weights by $e^{\alpha_m}$, correct ones by $e^{-\alpha_m}$**, then renormalize —
exactly the AdaBoost reweighting. So each step is one coordinate of greedy
descent on the exponential loss.

**SAMME (multiclass).** For $K$ classes with stumps predicting a label directly,
Zhu et al. show the analogous learner weight is

$$\alpha_m=\log\frac{1-\varepsilon_m}{\varepsilon_m}+\log(K-1).$$

The extra $\log(K-1)$ makes $\alpha_m>0$ whenever the learner beats *random*
guessing ($\varepsilon_m<1-\tfrac1K$), not merely $<\tfrac12$. For $K=2$ it
reduces to classic AdaBoost. The final classifier is the weighted plurality vote.

**Why it works / margins.** Minimizing exponential loss drives up the *margin*
$y_i F(x_i)$; AdaBoost keeps improving test error even after training error hits
zero because it keeps increasing margins (Schapire et al.). The flip side: the
exponential loss is **not robust to label noise** — outliers get exponentially
large weights.

## 4. NumPy implementation (SAMME with weighted-Gini decision stumps)

In [ ]:
# ===== actual implementation from adaboost.py =====
from __future__ import annotations

import numpy as np

SEED = 0

class _Stump:
    def __init__(self, n_classes):
        self.n_classes = n_classes
        self.feature = self.threshold = None
        self.left_class = self.right_class = None

    @staticmethod
    def _weighted_gini(w_per_class, w_total):
        if w_total <= 0:
            return 0.0
        p = w_per_class / w_total
        return 1.0 - (p ** 2).sum()

    def fit(self, X, y, w):
        X = np.asarray(X, float)
        y = np.asarray(y).astype(int)
        n, d = X.shape
        C = self.n_classes
        best = (np.inf, None, None, None, None)   # (impurity, f, t, lc, rc)
        for f in range(d):
            order = np.argsort(X[:, f], kind="mergesort")
            xs, ys, ws = X[order, f], y[order], w[order]
            # per-class weighted prefix sums -> evaluate every threshold at once
            oh = np.eye(C)[ys] * ws[:, None]          # (n, C) weighted one-hot
            cum = np.cumsum(oh, axis=0)               # left class-weight prefix
            tot = cum[-1]
            valid = xs[:-1] != xs[1:]
            for j in np.where(valid)[0]:
                wl = cum[j]                            # left class weights
                wr = tot - wl                          # right class weights
                gl, gr = wl.sum(), wr.sum()
                imp = (gl * self._weighted_gini(wl, gl)
                       + gr * self._weighted_gini(wr, gr)) / (gl + gr + 1e-12)
                if imp < best[0]:
                    best = (imp, f, (xs[j] + xs[j + 1]) / 2,
                            int(wl.argmax()), int(wr.argmax()))
        _, self.feature, self.threshold, self.left_class, self.right_class = best
        # degenerate fallback: no valid split found
        if self.feature is None:
            self.feature, self.threshold = 0, 0.0
            maj = int((np.eye(C)[y] * w[:, None]).sum(0).argmax())
            self.left_class = self.right_class = maj
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        return np.where(X[:, self.feature] <= self.threshold,
                        self.left_class, self.right_class)

class AdaBoostNumPy:
    """Discrete SAMME AdaBoost with decision-stump weak learners.

    For K=2 this reduces to classic Freund-Schapire AdaBoost.M1. The SAMME
    learner weight adds a $\\log(K-1)$ term so a learner only needs to beat
    *random* (accuracy > 1/K), not 1/2, to contribute positively.
    """

    def __init__(self, n_estimators=50, seed=SEED):
        self.n_estimators = n_estimators
        self.seed = seed
        self.stumps_ = []
        self.alphas_ = []
        self.errors_ = []

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y).astype(int)
        n = len(X)
        self.n_classes = int(y.max()) + 1
        K = self.n_classes
        w = np.full(n, 1.0 / n)            # uniform initial sample weights
        self.stumps_, self.alphas_, self.errors_ = [], [], []

        for _ in range(self.n_estimators):
            stump = _Stump(K).fit(X, y, w)
            pred = stump.predict(X)
            miss = (pred != y).astype(float)
            err = float(np.dot(w, miss) / w.sum())          # weighted error
            err = min(max(err, 1e-10), 1 - 1e-10)           # clip for stability

            # SAMME learner weight (derivation in the notebook):
            #   alpha = log((1-err)/err) + log(K-1)
            alpha = np.log((1 - err) / err) + np.log(K - 1)
            if alpha <= 0 and self.stumps_:
                # learner worse than random and we already have learners -> stop
                break

            # reweight: up-weight misclassified, down-weight correct, renormalize
            w = w * np.exp(alpha * miss)
            w = w / w.sum()

            self.stumps_.append(stump)
            self.alphas_.append(alpha)
            self.errors_.append(err)
        self.alphas_ = np.array(self.alphas_)
        return self

    def decision_function(self, X):
        """Aggregate weighted votes per class: (n, K) score matrix."""
        n = len(np.asarray(X, float))
        scores = np.zeros((n, self.n_classes))
        for stump, alpha in zip(self.stumps_, self.alphas_):
            pred = stump.predict(X)
            scores[np.arange(n), pred] += alpha          # each stump votes alpha
        return scores

    def predict(self, X):
        return self.decision_function(X).argmax(1)

    def staged_predict(self, X):
        """Yield the ensemble prediction after each added stump (for curves)."""
        n = len(np.asarray(X, float))
        scores = np.zeros((n, self.n_classes))
        for stump, alpha in zip(self.stumps_, self.alphas_):
            scores[np.arange(n), stump.predict(X)] += alpha
            yield scores.argmax(1)

## 5. Reference / cross-check — why not PyTorch?

AdaBoost stacks *discrete, non-differentiable* decision stumps and reweights
samples with a closed-form rule — there is no gradient to backpropagate, so an
idiomatic PyTorch model is not the natural tool. We cross-check the from-scratch
SAMME implementation against scikit-learn's `AdaBoostClassifier` (also SAMME).

In [ ]:
# ===== actual implementation from adaboost.py =====
def sklearn_reference(X, y, **kw):
    from sklearn.ensemble import AdaBoostClassifier
    from sklearn.tree import DecisionTreeClassifier
    kw.setdefault("n_estimators", 50)
    stump = DecisionTreeClassifier(max_depth=1)
    # sklearn >=1.6 dropped the `algorithm` arg (SAMME is now the only option);
    # older versions need algorithm="SAMME" to match our discrete boosting.
    try:
        return AdaBoostClassifier(estimator=stump, algorithm="SAMME",
                                  random_state=SEED, **kw).fit(X, y)
    except TypeError:
        return AdaBoostClassifier(estimator=stump,
                                  random_state=SEED, **kw).fit(X, y)

def demo():
    np.random.seed(SEED)
    from sklearn.datasets import make_classification

    # ---------- binary (K=2) ----------
    X, y = make_classification(n_samples=400, n_features=10, n_informative=5,
                               n_redundant=2, random_state=SEED)
    Xtr, ytr, Xte, yte = X[:300], y[:300], X[300:], y[300:]
    ada = AdaBoostNumPy(n_estimators=50).fit(Xtr, ytr)
    print(f"[bin] AdaBoost ({len(ada.stumps_)} stumps) acc="
          f"{np.mean(ada.predict(Xte) == yte):.3f}  "
          f"first/last err={ada.errors_[0]:.3f}/{ada.errors_[-1]:.3f}")
    print(f"[bin] single stump acc="
          f"{np.mean(_Stump(2).fit(Xtr, ytr, np.full(len(Xtr), 1/len(Xtr))).predict(Xte) == yte):.3f}")
    sk = sklearn_reference(Xtr, ytr, n_estimators=50)
    print(f"[bin] sklearn SAMME acc={np.mean(sk.predict(Xte) == yte):.3f}")

    # ---------- multiclass (K=3) via SAMME ----------
    Xm, ym = make_classification(n_samples=600, n_features=12, n_informative=8,
                                 n_classes=3, n_clusters_per_class=1, random_state=SEED)
    Xmtr, ymtr, Xmte, ymte = Xm[:450], ym[:450], Xm[450:], ym[450:]
    adam = AdaBoostNumPy(n_estimators=80).fit(Xmtr, ymtr)
    print(f"[mc ] SAMME ({len(adam.stumps_)} stumps) acc={np.mean(adam.predict(Xmte) == ymte):.3f}")
    skm = sklearn_reference(Xmtr, ymtr, n_estimators=80)
    print(f"[mc ] sklearn SAMME acc={np.mean(skm.predict(Xmte) == ymte):.3f}")

    # boosting curve: test accuracy improves as stumps accumulate
    accs = [np.mean(p == ymte) for p in adam.staged_predict(Xmte)]
    print(f"[mc ] staged acc @ 1/10/40/last: "
          f"{accs[0]:.3f}/{accs[9]:.3f}/{accs[39]:.3f}/{accs[-1]:.3f}")

## 6. Train — binary & multiclass AdaBoost vs the sklearn cross-check

In [ ]:
demo()

## 7. Visualization — test accuracy vs number of stumps; weight evolution

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
import adaboost as M

X, y = make_classification(n_samples=600, n_features=12, n_informative=8,
                           n_classes=3, n_clusters_per_class=1, random_state=0)
Xtr, ytr, Xte, yte = X[:450], y[:450], X[450:], y[450:]
ada = M.AdaBoostNumPy(n_estimators=120).fit(Xtr, ytr)

acc_curve = [np.mean(p == yte) for p in ada.staged_predict(Xte)]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(range(1, len(acc_curve) + 1), acc_curve)
ax[0].set_xlabel("number of stumps"); ax[0].set_ylabel("test accuracy")
ax[0].set_title("Boosting curve: weak learners -> strong ensemble")

ax[1].plot(ada.errors_, "o-", ms=3)
ax[1].axhline(1 - 1/3, ls="--", c="r", label="random (1 - 1/K)")
ax[1].set_xlabel("round"); ax[1].set_ylabel("weighted error of stump")
ax[1].set_title("Per-round weighted error stays < random"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- AdaBoost = forward stagewise fitting of the **exponential loss**; $\alpha_m$ and
  the reweighting both fall out of one line of algebra.
- **SAMME**'s $\log(K-1)$ term is what lets stumps contribute in the multiclass
  setting (beat $1/K$, not $1/2$).
- **Not robust to noise/outliers:** mislabeled points get exponentially
  up-weighted; consider gradient boosting with a robust loss (e.g. logistic /
  Huber) when labels are noisy.
- Weak base learners (stumps) are deliberate — strong base learners overfit fast
  under boosting. Stop when a learner can no longer beat random ($\alpha\le 0$).